# 02B - Emotion2Vec Downstream Fine-Tuning tren IEMOCAP 5-Fold va 10-Fold

Notebook nay la buoc rieng de fine-tune theo huong downstream cua Emotion2Vec truoc khi quay lai notebook 03/04.

Can hieu dung:

- Paper/repo Emotion2Vec downstream IEMOCAP thuong dung `emotion2vec` nhu SSL/pretrained feature extractor, sau do train downstream layers tren feature 768-D.
- Vi vay notebook 02B hien tai chay official-style downstream fine-tuning: train model nho truc tiep tren `emotion2vec` embedding, danh gia ca `5-fold session-independent` va `10-fold speaker-independent`.
- Neu muon full backbone fine-tune tu raw waveform, Kaggle dataset phai co `audio_wav/` va can them training API cho backbone FunASR/Emotion2Vec. Phan do nang hon va khong the lam chi tu `.npz` cache.

Output chinh:

```text
02B.../output/finetuned_models/
02B.../output/reports/
02B.../output/predictions/
02B.../output/02B_emotion2vec_finetune_outputs.zip
```

Sau buoc nay:

- Dung ket qua 02B de biet Emotion2Vec-only downstream manh toi dau.
- Neu 02B tot hon 03/04, ta se xem lai fusion/acoustic branch.
- Neu can raw-audio backbone fine-tune, ta se tao tiep mot notebook raw-waveform fine-tune rieng.

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

SEED = int(os.getenv("SEED", "42"))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
def zip_folder(source_dir, zip_path, exclude_dir_names=None, exclude_suffixes=None):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    exclude_dir_names = set(exclude_dir_names or [])
    exclude_suffixes = set(exclude_suffixes or [])
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in source_dir.rglob("*"):
            if not file_path.is_file():
                continue
            if any(part in exclude_dir_names for part in file_path.relative_to(source_dir).parts):
                continue
            if file_path.suffix.lower() in exclude_suffixes:
                continue
            zf.write(file_path, file_path.relative_to(source_dir))
    return zip_path

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "Speech Project" and PROJECT_ROOT.parent.name == "Speech Project":
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

BASE_DIR = PROJECT_ROOT / "06_w2v_based_models"
if not BASE_DIR.exists() and Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working") / "06_w2v_based_models"

LOCAL_DATA_DIR = PROJECT_ROOT / "06_w2v_based_models" / "data"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
ENV_DATA_DIR = os.environ.get("IEMOCAP_DATA_DIR", "").strip()
KNOWN_KAGGLE_DATASETS = [
    KAGGLE_INPUT_DIR / "multitask",
    KAGGLE_INPUT_DIR / "iemocap-full-multitask-data",
    KAGGLE_INPUT_DIR / "iemocap_full_multitask_data",
    KAGGLE_INPUT_DIR / "iemocap-multitask-train-data",
    KAGGLE_INPUT_DIR / "iemocap_multitask_train_data",
]

def unique_existing_paths(paths):
    seen = set()
    out = []
    for path in paths:
        if not path:
            continue
        path = Path(path)
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            out.append(path)
    return out

def search_roots():
    roots = []
    env_dir = os.environ.get("IEMOCAP_DATA_DIR", "").strip()
    if env_dir:
        roots.append(Path(env_dir))
    roots.extend([
        LOCAL_DATA_DIR,
        BASE_DIR / "data",
        BASE_DIR,
        PROJECT_ROOT,
        *KNOWN_KAGGLE_DATASETS,
        KAGGLE_INPUT_DIR,
    ])
    return unique_existing_paths(roots)

def rank_candidate(path):
    text = str(path).replace("\\", "/").lower()
    preferred = 0
    if ENV_DATA_DIR:
        try:
            env_text = str(Path(ENV_DATA_DIR).resolve()).replace("\\", "/").lower()
            if text.startswith(env_text):
                preferred -= 20
        except Exception:
            pass
    if "/multitask/" in text or text.endswith("/multitask"):
        preferred -= 4
    if "/output/" in text:
        preferred -= 3
    if "/data/" in text:
        preferred -= 2
    if "/output/features/" in text:
        preferred -= 2
    if "/features/" in text:
        preferred -= 1
    return (preferred, len(path.parts), text)

def find_named_file(filename, env_var=None, extra_candidates=None, require_exists=True, description=None):
    candidates = []
    if env_var:
        env_value = os.environ.get(env_var, "").strip()
        if env_value:
            candidates.append(Path(env_value))
    if extra_candidates:
        candidates.extend(Path(p) for p in extra_candidates)

    for root in search_roots():
        candidates.extend([
            root / filename,
            root / "data" / filename,
            root / "metadata" / filename,
            root / "splits" / filename,
            root / "features" / filename,
            root / "output" / filename,
            root / "output" / "metadata" / filename,
            root / "output" / "splits" / filename,
            root / "output" / "features" / filename,
        ])
        if root.exists():
            try:
                candidates.extend(root.rglob(filename))
            except Exception:
                pass

    existing = [Path(p).resolve() for p in candidates if Path(p).exists() and Path(p).is_file()]
    if existing:
        existing = sorted(set(existing), key=rank_candidate)
        return existing[0]

    if require_exists:
        roots_text = "\n".join(f"- {root}" for root in search_roots())
        raise FileNotFoundError(
            f"Không tìm thấy {description or filename}. Notebook đã quét các root:\n{roots_text}\n"
            f"Nếu file nằm chỗ khác, hãy set {env_var or 'biến môi trường tương ứng'}."
        )
    return None

def has_train_tables(path):
    return (
        path is not None
        and (path / "metadata" / "iemocap_metadata_full.csv").exists()
        and (path / "splits").exists()
    )

def candidate_data_roots(base):
    if base is None:
        return []
    base = Path(base)
    roots = [base, base / "data"]
    if base.exists():
        for meta_path in base.rglob("iemocap_metadata_full.csv"):
            roots.append(meta_path.parent.parent)
    return roots

def resolve_metadata_path(require_exists=False):
    return find_named_file(
        "iemocap_metadata_full.csv",
        env_var="IEMOCAP_METADATA_PATH",
        require_exists=require_exists,
        description="metadata IEMOCAP `iemocap_metadata_full.csv`",
    )

def resolve_split_path(split_file):
    return find_named_file(
        split_file,
        env_var="IEMOCAP_SPLIT_PATH",
        description=f"split file `{split_file}`",
    )

def resolve_fallback_data_dir():
    roots = search_roots()
    if roots:
        return roots[0].resolve()
    return PROJECT_ROOT.resolve()

FULL_METADATA_PATH = resolve_metadata_path(require_exists=False)
if FULL_METADATA_PATH is not None:
    METADATA_DIR = FULL_METADATA_PATH.parent
    DATA_DIR = METADATA_DIR.parent if METADATA_DIR.name == "metadata" else METADATA_DIR
else:
    DATA_DIR = resolve_fallback_data_dir()
    METADATA_DIR = DATA_DIR / "metadata"
SPLIT_DIR = None
AUDIO_DIR = DATA_DIR / "audio_wav"

WORKING_DATA_DIR = Path("/kaggle/working/iemocap_full_multitask_data") if Path("/kaggle/working").exists() else DATA_DIR
WORKING_FEATURE_DIR = WORKING_DATA_DIR / "features"
WORKING_REPORT_DIR = WORKING_DATA_DIR / "feature_reports"
WORKING_FIGURE_DIR = WORKING_DATA_DIR / "feature_figures"
INPUT_FEATURE_DIR = DATA_DIR / "features"
FEATURE_CACHE_NAME = "iemocap_full_emotion2vec_acoustic_cache.npz"
FINETUNED_FEATURE_CACHE_NAME = os.getenv("FINETUNED_FEATURE_CACHE_NAME", "iemocap_finetuned_emotion2vec_acoustic_cache.npz")
WORKING_FEATURE_CACHE_PATH = WORKING_FEATURE_DIR / FEATURE_CACHE_NAME
INPUT_FEATURE_CACHE_PATH = INPUT_FEATURE_DIR / FEATURE_CACHE_NAME

def resolve_feature_cache_path(require_exists=False):
    env_cache = os.environ.get("IEMOCAP_FEATURE_CACHE", "").strip()
    candidates = []
    if env_cache:
        candidates.append(Path(env_cache))
    candidates.extend([
        WORKING_FEATURE_DIR / FINETUNED_FEATURE_CACHE_NAME,
        INPUT_FEATURE_DIR / FINETUNED_FEATURE_CACHE_NAME,
        DATA_DIR / "output" / "features" / FINETUNED_FEATURE_CACHE_NAME,
        WORKING_FEATURE_CACHE_PATH,
        INPUT_FEATURE_CACHE_PATH,
        DATA_DIR / "output" / "features" / FEATURE_CACHE_NAME,
    ])
    for root in search_roots():
        candidates.extend([
            root / FINETUNED_FEATURE_CACHE_NAME,
            root / "features" / FINETUNED_FEATURE_CACHE_NAME,
            root / "data" / "features" / FINETUNED_FEATURE_CACHE_NAME,
            root / "output" / "features" / FINETUNED_FEATURE_CACHE_NAME,
            root / FEATURE_CACHE_NAME,
            root / "features" / FEATURE_CACHE_NAME,
            root / "data" / "features" / FEATURE_CACHE_NAME,
            root / "output" / "features" / FEATURE_CACHE_NAME,
        ])
        try:
            candidates.extend(root.rglob(FINETUNED_FEATURE_CACHE_NAME))
            candidates.extend(root.rglob(FEATURE_CACHE_NAME))
        except Exception:
            pass

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    if require_exists:
        raise FileNotFoundError(
            "Không tìm thấy full feature cache. Notebook đã tự quét `data/features/`, `output/features/` "
            "và toàn bộ `/kaggle/input`. Nếu bạn dùng Kaggle dataset `quanghuy225/multitask`, hãy đảm bảo file "
            f"`{FEATURE_CACHE_NAME}` nằm trong dataset, ví dụ `/kaggle/input/multitask/output/features/{FEATURE_CACHE_NAME}`."
        )
    return WORKING_FEATURE_CACHE_PATH

print("DATA_DIR:", DATA_DIR)
print("FULL_METADATA_PATH:", FULL_METADATA_PATH, bool(FULL_METADATA_PATH and FULL_METADATA_PATH.exists()))
print("AUDIO_DIR:", AUDIO_DIR, AUDIO_DIR.exists())
print("INPUT_FEATURE_CACHE_PATH:", INPUT_FEATURE_CACHE_PATH, INPUT_FEATURE_CACHE_PATH.exists())
print("WORKING_FEATURE_CACHE_PATH:", WORKING_FEATURE_CACHE_PATH, WORKING_FEATURE_CACHE_PATH.exists())
print("WORKING_REPORT_DIR:", WORKING_REPORT_DIR)
print("WORKING_FIGURE_DIR:", WORKING_FIGURE_DIR)

In [ ]:
EMOTIONS = ["neutral", "angry", "sad", "happy"]
ID_TO_EMOTION = {0: "neutral", 1: "angry", 2: "sad", 3: "happy"}

EPOCHS = int(os.getenv("EPOCHS", "45"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))
LR = float(os.getenv("LR", "5e-4"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "2e-4"))
PATIENCE = int(os.getenv("PATIENCE", "10"))
HIDDEN_DIM = int(os.getenv("HIDDEN_DIM", "256"))
DROPOUT = float(os.getenv("DROPOUT", "0.30"))
N_SEEDS = int(os.getenv("N_SEEDS", "1"))
USE_WEIGHTED_SAMPLER = os.getenv("USE_WEIGHTED_SAMPLER", "1") == "1"
LOSS_EMOTION_WEIGHT = float(os.getenv("LOSS_EMOTION_WEIGHT", "1.35"))
LOSS_AVD_WEIGHT = float(os.getenv("LOSS_AVD_WEIGHT", "0.30"))
LOSS_CCC_WEIGHT = float(os.getenv("LOSS_CCC_WEIGHT", "0.55"))
PRIMARY_SCORE_MODE = os.getenv("PRIMARY_SCORE_MODE", "all").lower()

def normalize_avd(df):
    out = df.copy()
    for src, dst in [("valence", "valence_norm"), ("arousal", "arousal_norm"), ("dominance", "dominance_norm")]:
        if dst not in out.columns:
            out[dst] = (pd.to_numeric(out[src], errors="coerce") - 1.0) / 4.0
        out[dst] = out[dst].clip(0.0, 1.0)
    return out

def load_feature_cache(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy full feature cache: {path}. Hãy chạy notebook 02 trước, hoặc upload file .npz này vào data/features/."
        )
    cache = np.load(path, allow_pickle=True)
    return {
        "sample_ids": cache["sample_ids"].astype(str),
        "emotion2vec": cache["emotion2vec"].astype(np.float32),
        "acoustic": cache["acoustic"].astype(np.float32),
        "acoustic_names": cache["acoustic_names"].astype(str) if "acoustic_names" in cache else np.array([]),
    }

def load_and_align_split(split_path, metadata_path, cache):
    split_df = pd.read_csv(split_path)
    required_cols = ["train_sample_id", "fold", "split", "emotion_id", "valence", "arousal", "dominance"]
    missing_required = [c for c in required_cols if c not in split_df.columns]
    if missing_required:
        raise ValueError(
            f"Split file thiếu các cột bắt buộc để train: {missing_required}. "
            "Notebook 03/04 cần split long CSV có sẵn label emotion_id, valence, arousal, dominance."
        )

    if metadata_path is not None and Path(metadata_path).exists():
        meta = pd.read_csv(metadata_path)
        extra_cols = ["train_sample_id", "transcription", "source_filename", "duration", "sample_rate", "channels"]
        extra_cols = [c for c in extra_cols if c in meta.columns and c not in split_df.columns]
        if extra_cols:
            split_df = split_df.merge(
                meta[["train_sample_id", *extra_cols]].drop_duplicates("train_sample_id"),
                on="train_sample_id",
                how="left",
            )
    else:
        print("Metadata full không có trong Kaggle input. Tiếp tục train bằng split long CSV vì split đã có label.")

    split_df = normalize_avd(split_df)

    id_to_idx = {sid: i for i, sid in enumerate(cache["sample_ids"])}
    split_df["feature_idx"] = split_df["train_sample_id"].astype(str).map(id_to_idx)
    missing = split_df["feature_idx"].isna().sum()
    if missing:
        missing_ids = split_df.loc[split_df["feature_idx"].isna(), "train_sample_id"].head(10).tolist()
        raise ValueError(f"{missing} rows have no feature cache. First missing ids: {missing_ids}")
    split_df["feature_idx"] = split_df["feature_idx"].astype(int)
    return split_df

class FullFeatureDataset(Dataset):
    def __init__(self, df, cache, e_scaler=None, a_scaler=None, fit_scalers=False):
        self.df = df.reset_index(drop=True).copy()
        idx = self.df["feature_idx"].values
        e = cache["emotion2vec"][idx]
        a = cache["acoustic"][idx]
        if fit_scalers:
            self.e_scaler = StandardScaler().fit(e)
            self.a_scaler = StandardScaler().fit(a)
        else:
            self.e_scaler = e_scaler
            self.a_scaler = a_scaler
        self.e = torch.tensor(self.e_scaler.transform(e), dtype=torch.float32)
        self.a = torch.tensor(self.a_scaler.transform(a), dtype=torch.float32)
        self.y_cls = torch.tensor(self.df["emotion_id"].values, dtype=torch.long)
        self.y_reg = torch.tensor(self.df[["valence_norm", "arousal_norm", "dominance_norm"]].values, dtype=torch.float32)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return {"e2v": self.e[idx], "acoustic": self.a[idx], "emotion": self.y_cls[idx], "avd": self.y_reg[idx], "idx": idx}

class MLPBranch(nn.Module):
    def __init__(self, in_dim, out_dim=256, hidden=512, dropout=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
    def forward(self, x):
        return self.net(x)

class Emotion2VecAcousticCoAttentionMultiTask(nn.Module):
    """
    Full proposed model:
      Branch A: frozen emotion2vec embedding -> adapter MLP -> z_e
      Branch B: handcrafted acoustic vector -> acoustic MLP -> z_a
      Cross attention: acoustic query attends to emotion2vec and emotion2vec query attends to acoustic
      Fusion: gated fusion -> shared representation
      Head 1: 4-class emotion classification
      Head 2: valence/arousal/dominance regression
    """
    def __init__(self, e2v_dim, acoustic_dim, hidden=256, heads=4, dropout=0.30, n_classes=4):
        super().__init__()
        self.e2v_branch = MLPBranch(e2v_dim, out_dim=hidden, hidden=hidden * 2, dropout=dropout)
        self.acoustic_branch = MLPBranch(acoustic_dim, out_dim=hidden, hidden=hidden * 2, dropout=dropout)
        self.a_queries_e = nn.MultiheadAttention(hidden, heads, dropout=0.1, batch_first=True)
        self.e_queries_a = nn.MultiheadAttention(hidden, heads, dropout=0.1, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.Sigmoid())
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 4, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.emotion_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )
        self.avd_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 3),
            nn.Sigmoid(),
        )
    def forward(self, e2v, acoustic):
        z_e = self.e2v_branch(e2v)
        z_a = self.acoustic_branch(acoustic)
        e_tok = z_e.unsqueeze(1)
        a_tok = z_a.unsqueeze(1)
        a_ctx, _ = self.a_queries_e(query=a_tok, key=e_tok, value=e_tok)
        e_ctx, _ = self.e_queries_a(query=e_tok, key=a_tok, value=a_tok)
        parts = torch.cat([z_e, z_a, e_ctx.squeeze(1), a_ctx.squeeze(1)], dim=-1)
        g = self.gate(parts)
        fused = self.fusion(parts)
        z = g * fused + (1.0 - g) * 0.5 * (z_e + z_a)
        return self.emotion_head(z), self.avd_head(z)

def ccc_torch(pred, target, eps=1e-8):
    pred_mean = torch.mean(pred, dim=0)
    target_mean = torch.mean(target, dim=0)
    pred_var = torch.var(pred, dim=0, unbiased=False)
    target_var = torch.var(target, dim=0, unbiased=False)
    cov = torch.mean((pred - pred_mean) * (target - target_mean), dim=0)
    return (2.0 * cov) / (pred_var + target_var + (pred_mean - target_mean) ** 2 + eps)

def ccc_loss(pred, target):
    return 1.0 - ccc_torch(pred, target).mean()

def ccc_np(pred, target, eps=1e-8):
    vals = []
    for i in range(pred.shape[1]):
        x, y = pred[:, i], target[:, i]
        mx, my = x.mean(), y.mean()
        vx, vy = x.var(), y.var()
        cov = np.mean((x - mx) * (y - my))
        vals.append((2 * cov) / (vx + vy + (mx - my) ** 2 + eps))
    return np.asarray(vals)

def class_weights(train_df):
    counts = train_df["emotion_id"].value_counts().sort_index()
    total = counts.sum()
    return torch.tensor([total / (4.0 * counts.get(i, 1)) for i in range(4)], dtype=torch.float32)

def make_train_sampler(train_df):
    counts = train_df["emotion_id"].value_counts().to_dict()
    weights = train_df["emotion_id"].map(lambda x: 1.0 / max(counts.get(x, 1), 1)).values
    return WeightedRandomSampler(
        weights=torch.tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
    )

def run_epoch(model, loader, optimizer, ce):
    model.train()
    losses = []
    for batch in loader:
        e = batch["e2v"].to(DEVICE)
        a = batch["acoustic"].to(DEVICE)
        yc = batch["emotion"].to(DEVICE)
        yr = batch["avd"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, pred = model(e, a)
        loss = LOSS_EMOTION_WEIGHT * ce(logits, yc) + LOSS_AVD_WEIGHT * nn.functional.smooth_l1_loss(pred, yr) + LOSS_CCC_WEIGHT * ccc_loss(pred, yr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def predict(model, loader):
    model.eval()
    logits, reg, yc, yr = [], [], [], []
    for batch in loader:
        lo, pr = model(batch["e2v"].to(DEVICE), batch["acoustic"].to(DEVICE))
        logits.append(lo.cpu().numpy())
        reg.append(pr.cpu().numpy())
        yc.append(batch["emotion"].numpy())
        yr.append(batch["avd"].numpy())
    return {"logits": np.concatenate(logits), "reg": np.concatenate(reg), "y_cls": np.concatenate(yc), "y_reg": np.concatenate(yr)}

def compute_metrics(pred):
    y_true = pred["y_cls"]
    y_pred = pred["logits"].argmax(axis=1)
    ccc = ccc_np(pred["reg"], pred["y_reg"])
    mae = np.mean(np.abs(pred["reg"] - pred["y_reg"]), axis=0)
    rmse = np.sqrt(np.mean((pred["reg"] - pred["y_reg"]) ** 2, axis=0))
    return {
        "WA": accuracy_score(y_true, y_pred),
        "UAR": balanced_accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "CCC_valence": ccc[0],
        "CCC_arousal": ccc[1],
        "CCC_dominance": ccc[2],
        "CCC_mean": float(ccc.mean()),
        "MAE_mean": float(mae.mean()),
        "RMSE_mean": float(rmse.mean()),
    }

def primary_score(m):
    if PRIMARY_SCORE_MODE == "emotion":
        return float(0.40 * m["WA"] + 0.40 * m["UAR"] + 0.20 * m["Macro_F1"])
    if PRIMARY_SCORE_MODE == "ccc":
        return float(m["CCC_mean"])
    return float(0.25 * m["WA"] + 0.30 * m["UAR"] + 0.25 * m["Macro_F1"] + 0.20 * m["CCC_mean"])

def resolve_warm_start_checkpoint():
    env_ckpt = os.getenv("MULTITASK_WARM_START_CHECKPOINT", "").strip()
    if env_ckpt and Path(env_ckpt).exists():
        return Path(env_ckpt).resolve()
    names = [
        "multitask_coattention_warm_start.pt",
        "03_04_warm_start.pt",
        "coattention_warm_start.pt",
    ]
    for root in search_roots():
        for name in names:
            direct_candidates = [
                root / name,
                root / "models" / name,
                root / "output" / "models" / name,
                root / "finetuned_models" / name,
            ]
            for candidate in direct_candidates:
                if candidate.exists():
                    return candidate.resolve()
            try:
                found = sorted(root.rglob(name))
            except Exception:
                found = []
            if found:
                return found[0].resolve()
    return None

def load_warm_start_if_available(model):
    ckpt_path = resolve_warm_start_checkpoint()
    if ckpt_path is None:
        return None
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt)) if isinstance(ckpt, dict) else ckpt
    missing, unexpected = model.load_state_dict(state, strict=False)
    print("Loaded warm-start checkpoint:", ckpt_path)
    print("Warm-start missing keys:", len(missing), "unexpected keys:", len(unexpected))
    return ckpt_path

In [ ]:
NOTEBOOK_DIR = BASE_DIR / "02BEmotion2Vec Downstream Finetune 5Fold 10Fold"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
REPORT_DIR = OUTPUT_DIR / "reports"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
MODEL_DIR = OUTPUT_DIR / "finetuned_models"
FIGURE_DIR = OUTPUT_DIR / "figures"
for folder in [OUTPUT_DIR, REPORT_DIR, PREDICTION_DIR, MODEL_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FEATURE_CACHE_PATH = resolve_feature_cache_path(require_exists=True)
PROTOCOLS = [
    ("5fold_session", "iemocap_5fold_session_long.csv", 5),
    ("10fold_speaker", "iemocap_10fold_speaker_long.csv", 10),
]
print("FEATURE_CACHE_PATH:", FEATURE_CACHE_PATH, FEATURE_CACHE_PATH.exists())
print("FULL_METADATA_PATH:", FULL_METADATA_PATH, bool(FULL_METADATA_PATH and FULL_METADATA_PATH.exists()))
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Audio raw available:", AUDIO_DIR, AUDIO_DIR.exists())

## Kien truc 02B

Notebook 02B chi dung nhanh `emotion2vec` de kiem tra suc manh cua pretrained representation truoc khi fusion acoustic o 03/04:

```text
emotion2vec embedding [B, 768]
 -> MLP adapter / downstream fine-tune head
 -> shared z [B, HIDDEN_DIM]
 -> Head 1: emotion classification
 -> Head 2: valence/arousal/dominance regression
```

Muc tieu cua 02B:

- Neu 02B emotion-only da manh hon 03/04, van de co the nam o fusion/acoustic branch.
- Neu 02B thap nhung 03/04 cao hon, acoustic branch dang giup ich.
- Neu ca hai thap, can chuyen sang raw-audio backbone fine-tune hoac frame-level/token-level Emotion2Vec thay vi utterance embedding.

In [ ]:
class Emotion2VecOnlyDataset(Dataset):
    def __init__(self, df, cache, scaler=None, fit_scaler=False):
        self.df = df.reset_index(drop=True).copy()
        x = cache["emotion2vec"][self.df["feature_idx"].values]
        if fit_scaler:
            self.scaler = StandardScaler().fit(x)
        else:
            self.scaler = scaler
        self.x = torch.tensor(self.scaler.transform(x), dtype=torch.float32)
        self.y_cls = torch.tensor(self.df["emotion_id"].values, dtype=torch.long)
        self.y_reg = torch.tensor(self.df[["valence_norm", "arousal_norm", "dominance_norm"]].values, dtype=torch.float32)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return {"x": self.x[idx], "emotion": self.y_cls[idx], "avd": self.y_reg[idx]}

class Emotion2VecDownstreamMultiTask(nn.Module):
    def __init__(self, input_dim=768, hidden=256, dropout=0.30, n_classes=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        self.emotion_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )
        self.avd_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 3),
            nn.Sigmoid(),
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.emotion_head(z), self.avd_head(z)

def train_epoch_e2v(model, loader, optimizer, ce):
    model.train()
    losses = []
    for batch in loader:
        x = batch["x"].to(DEVICE)
        yc = batch["emotion"].to(DEVICE)
        yr = batch["avd"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, pred = model(x)
        loss = LOSS_EMOTION_WEIGHT * ce(logits, yc) + LOSS_AVD_WEIGHT * nn.functional.smooth_l1_loss(pred, yr) + LOSS_CCC_WEIGHT * ccc_loss(pred, yr)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def predict_e2v(model, loader):
    model.eval()
    logits, reg, yc, yr = [], [], [], []
    for batch in loader:
        lo, pr = model(batch["x"].to(DEVICE))
        logits.append(lo.cpu().numpy())
        reg.append(pr.cpu().numpy())
        yc.append(batch["emotion"].numpy())
        yr.append(batch["avd"].numpy())
    return {"logits": np.concatenate(logits), "reg": np.concatenate(reg), "y_cls": np.concatenate(yc), "y_reg": np.concatenate(yr)}

In [ ]:
def run_protocol(protocol, split_file, n_expected_folds):
    split_path = resolve_split_path(split_file)
    cache = load_feature_cache(FEATURE_CACHE_PATH)
    split_df = load_and_align_split(split_path, FULL_METADATA_PATH, cache)
    if split_df["fold"].nunique() != n_expected_folds:
        raise ValueError(f"{protocol}: expected {n_expected_folds} folds, got {split_df['fold'].nunique()}")

    protocol_model_dir = MODEL_DIR / protocol
    protocol_report_dir = REPORT_DIR / protocol
    protocol_prediction_dir = PREDICTION_DIR / protocol
    for folder in [protocol_model_dir, protocol_report_dir, protocol_prediction_dir]:
        folder.mkdir(parents=True, exist_ok=True)

    rows = []
    preds = []
    for fold in sorted(split_df["fold"].unique()):
        print(f"Running {protocol}: {fold}")
        train_df = split_df[(split_df["fold"].eq(fold)) & (split_df["split"].eq("train"))].copy()
        val_df = split_df[(split_df["fold"].eq(fold)) & (split_df["split"].eq("validation"))].copy()
        test_df = split_df[(split_df["fold"].eq(fold)) & (split_df["split"].eq("test"))].copy()

        train_ds = Emotion2VecOnlyDataset(train_df, cache, fit_scaler=True)
        val_ds = Emotion2VecOnlyDataset(val_df, cache, scaler=train_ds.scaler)
        test_ds = Emotion2VecOnlyDataset(test_df, cache, scaler=train_ds.scaler)
        sampler = make_train_sampler(train_df) if USE_WEIGHTED_SAMPLER else None
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=(sampler is None), sampler=sampler, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        seed_preds = []
        histories = []
        seed_states = []
        best_epochs = []
        for seed_offset in range(N_SEEDS):
            seed_value = SEED + seed_offset
            set_seed(seed_value)
            model = Emotion2VecDownstreamMultiTask(input_dim=cache["emotion2vec"].shape[1], hidden=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=max(2, PATIENCE // 3))
            ce = nn.CrossEntropyLoss(weight=class_weights(train_df).to(DEVICE), label_smoothing=0.05)
            best_state = None
            best_score = -1e9
            best_epoch = 0
            patience_left = PATIENCE
            for epoch in range(1, EPOCHS + 1):
                loss = train_epoch_e2v(model, train_loader, optimizer, ce)
                val_pred = predict_e2v(model, val_loader)
                val_metrics = compute_metrics(val_pred)
                score = primary_score(val_metrics)
                scheduler.step(score)
                histories.append({"protocol": protocol, "fold": fold, "seed": seed_value, "epoch": epoch, "train_loss": loss, "primary_score": score, **{f"val_{k}": v for k, v in val_metrics.items()}})
                if score > best_score:
                    best_score = score
                    best_epoch = epoch
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    patience_left = PATIENCE
                else:
                    patience_left -= 1
                if patience_left <= 0:
                    break
            model.load_state_dict(best_state)
            seed_preds.append(predict_e2v(model, test_loader))
            seed_states.append({"seed": seed_value, "state_dict": best_state, "best_epoch": best_epoch, "best_score": best_score})
            best_epochs.append(best_epoch)

        test_pred = {
            "logits": np.mean([p["logits"] for p in seed_preds], axis=0),
            "reg": np.mean([p["reg"] for p in seed_preds], axis=0),
            "y_cls": seed_preds[0]["y_cls"],
            "y_reg": seed_preds[0]["y_reg"],
        }
        metrics = compute_metrics(test_pred)
        pred_cls = test_pred["logits"].argmax(axis=1)
        probs = torch.softmax(torch.tensor(test_pred["logits"]), dim=1).numpy()
        out = test_df.reset_index(drop=True).copy()
        out["pred_emotion_id"] = pred_cls
        out["pred_emotion"] = [ID_TO_EMOTION[int(x)] for x in pred_cls]
        for i, name in ID_TO_EMOTION.items():
            out[f"prob_{name}"] = probs[:, i]
        out["pred_valence"] = test_pred["reg"][:, 0] * 4.0 + 1.0
        out["pred_arousal"] = test_pred["reg"][:, 1] * 4.0 + 1.0
        out["pred_dominance"] = test_pred["reg"][:, 2] * 4.0 + 1.0

        safe = str(fold).replace(" ", "_").replace("/", "_")
        pd.DataFrame(histories).to_csv(protocol_report_dir / f"{safe}_history.csv", index=False, encoding="utf-8-sig")
        out.to_csv(protocol_prediction_dir / f"{safe}_predictions.csv", index=False, encoding="utf-8-sig")
        torch.save({"seed_states": seed_states, "fold": fold, "best_epochs": best_epochs, "metrics": metrics, "scaler": train_ds.scaler}, protocol_model_dir / f"{safe}_emotion2vec_downstream.pt")
        row = {"protocol": protocol, "fold": fold, "best_epoch": int(np.max(best_epochs)), "n_seeds": N_SEEDS, "n_train": len(train_df), "n_validation": len(val_df), "n_test": len(test_df), **metrics}
        rows.append(row)
        preds.append(out)
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items() if k in ["fold", "WA", "UAR", "Macro_F1", "CCC_mean"]})

    metrics_df = pd.DataFrame(rows)
    all_preds = pd.concat(preds, ignore_index=True)
    metrics_df.to_csv(protocol_report_dir / f"{protocol}_02B_metrics.csv", index=False, encoding="utf-8-sig")
    all_preds.to_csv(protocol_prediction_dir / f"{protocol}_02B_predictions.csv", index=False, encoding="utf-8-sig")
    return metrics_df

In [ ]:
all_metric_tables = []
start = time.time()
for protocol, split_file, n_folds in PROTOCOLS:
    metrics_df = run_protocol(protocol, split_file, n_folds)
    all_metric_tables.append(metrics_df)
print("Seconds:", round(time.time() - start, 2))
all_metrics = pd.concat(all_metric_tables, ignore_index=True)
all_metrics.to_csv(REPORT_DIR / "02B_all_protocol_metrics.csv", index=False, encoding="utf-8-sig")
display(all_metrics)
summary = all_metrics.drop(columns=["fold", "protocol"], errors="ignore").select_dtypes(include=[np.number]).agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})
summary.to_csv(REPORT_DIR / "02B_all_protocol_summary.csv", index=False, encoding="utf-8-sig")
display(summary)

## Zip output 02B

Cell cuoi gom toan bo checkpoint, report va prediction cua 02B. Giu lai zip nay neu muon so sanh hoac dung lai downstream model ve sau.

In [ ]:
zip_path = Path("/kaggle/working") / "02B_emotion2vec_finetune_outputs.zip" if Path("/kaggle/working").exists() else OUTPUT_DIR / "02B_emotion2vec_finetune_outputs.zip"
zip_folder(OUTPUT_DIR, zip_path)
print("ZIP_OUTPUT:", zip_path)
print("ZIP_SIZE_MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))